# SAM Predictions vs Ground Truth Annotations

This notebook evaluates Segment Anything Model (SAM) predictions against COCO ground truth masks.

## Evaluation Metrics:
- **IoU (Intersection over Union)**: Overlap between predicted and ground truth masks
- **Dice Coefficient**: Similarity measure for segmentation
- **Precision & Recall**: Per-pixel classification accuracy
- **Boundary F1 Score**: Edge quality metric
- **Mean metrics across dataset samples**

## 1. Install Dependencies

In [ ]:
!pip install segment-anything git+https://github.com/facebookresearch/segment-anything.git
!pip install opencv-python matplotlib pycocotools scikit-image pandas seaborn

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-zrc97g6b
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-zrc97g6b
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36592 sha256=8a02c58b22309fe0002bd368197eacc1edc010f2e79a3db6ab9139e43f3fcbbe
  Stored in directory: /tmp/pip-ephem-wheel-cache-duahkxqs/wheels/29/82/ff/04e2be9805a1cb48bec0b85b5a6da6b63f647645750a0e42d4
Successfully built segment_anything


## 2. Import Libraries

In [ ]:
import torch
import numpy as np
import cv2
from segment_anything import sam_model_registry, SamPredictor
import urllib.request
import os
import pandas as pd
import json
from pathlib import Path
from pycocotools import mask as mask_utils
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

# Paths
image_dir = '/content/drive/MyDrive/sam_dataset/extracted/'
output_dir = '/content/drive/MyDrive/marl/'


Mounted at /content/drive


## 3. Download SAM Model

In [ ]:
# !nvidia-smi
# Choose model size: 'vit_h' (best), 'vit_l', or 'vit_b' (fastest)
model_type = "vit_b"

checkpoint_url = {
    'vit_h': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
    'vit_l': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
    'vit_b': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
}

checkpoint_path = f"sam_{model_type}.pth"

if not os.path.exists(checkpoint_path):
    print(f"Downloading {model_type} checkpoint...")
    urllib.request.urlretrieve(checkpoint_url[model_type], checkpoint_path)
    print("Download complete!")

# Initialize SAM
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
sam.to(device=device)
predictor = SamPredictor(sam)

print("SAM model loaded!")

Mon Dec  1 12:45:05 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
def calculate_iou(pred_mask, gt_mask):
    """Calculate Intersection over Union."""
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    if union == 0:
        return 0.0
    return intersection / union

def calculate_dice(pred_mask, gt_mask):
    """Calculate Dice coefficient."""
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    total = pred_mask.sum() + gt_mask.sum()
    if total == 0:
        return 0.0
    return 2 * intersection / total

def calculate_metrics(pred_mask, gt_mask):
    """Calculate comprehensive metrics."""
    iou = calculate_iou(pred_mask, gt_mask)
    dice = calculate_dice(pred_mask, gt_mask)

    # intersection = np.logical_and(pred_mask, gt_mask).sum()
    # pred_area = pred_mask.sum()
    # gt_area = gt_mask.sum()

    # precision = intersection / pred_area if pred_area > 0 else 0
    # recall = intersection / gt_area if gt_area > 0 else 0
    # f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'iou': iou,
        'dice': dice
    }

In [ ]:
def decode_sa1b_mask(annotation):
    """Decode SA-1B RLE mask to binary mask."""
    segmentation = annotation['segmentation']
    if isinstance(segmentation, dict):
        mask = mask_utils.decode(segmentation)
    else:
        raise ValueError(f"Unexpected segmentation format: {type(segmentation)}")
    return mask.astype(bool)


def load_sa1b_image_and_annotations(json_path):
    """Load image and annotations from SA-1B format."""
    json_path = Path(json_path)

    # Load JSON
    with open(json_path, 'r') as f:
        data = json.load(f)

    # Load image
    img_path = json_path.with_suffix('.jpg')
    if not img_path.exists():
        raise FileNotFoundError(f"Image not found: {img_path}")

    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    return image, data


def evaluate_sam_optimized(extracted_dir, predictor, box_batch_size=32):
    """Optimized with proper batching."""
    extracted_dir = Path(extracted_dir)
    json_files = list(extracted_dir.glob("**/*.json"))
    json_files.sort()
    json_files = json_files[5300 + 11165::]

    results = []
    file_no = 165

    for idx, json_path in enumerate(tqdm(json_files)):
        try:
            # Load image and annotations
            image, data = load_sa1b_image_and_annotations(json_path)
            predictor.set_image(image)

            annotations = data['annotations']
            gt_masks = [decode_sa1b_mask(ann) for ann in annotations]
            boxes = np.array([[ann['bbox'][0], ann['bbox'][1],
                              ann['bbox'][0] + ann['bbox'][2],
                              ann['bbox'][1] + ann['bbox'][3]]
                             for ann in annotations])
            annotation_ids = [ann['id'] for ann in annotations]

            # Process boxes in batches
            all_pred_masks = []
            for batch_start in range(0, len(boxes), box_batch_size):
                batch_end = min(batch_start + box_batch_size, len(boxes))
                batch_boxes = boxes[batch_start:batch_end]

                with torch.no_grad():
                    # Process one box at a time within batch
                    batch_masks = []
                    for box in batch_boxes:
                        masks, _, _ = predictor.predict(
                            box=box[None, :],  # Add batch dimension
                            multimask_output=False,
                        )
                        batch_masks.append(masks[0])

                    all_pred_masks.extend(batch_masks)

            # Calculate metrics
            image_file = json_path.stem + ".jpg"
            for jdx, (pred_mask, gt_mask) in enumerate(zip(all_pred_masks, gt_masks)):
                pred_bool = pred_mask.astype(bool)
                gt_bool = gt_mask.astype(bool)

                inter = np.logical_and(pred_bool, gt_bool).sum()
                union = np.logical_or(pred_bool, gt_bool).sum()

                results.append({
                    'image_file': image_file,
                    'annotation': annotation_ids[jdx],
                    'IoU': float(inter / (union + 1e-10)),
                    'Dice': float(2 * inter / (pred_bool.sum() + gt_bool.sum() + 1e-10)),
                })

            predictor.reset_image()

            # Save checkpoint every 100 images
            if (idx + 1) % 100 == 0:
                pd.DataFrame(results).to_csv(
                    f'/content/drive/MyDrive/sam_dataset/metrics/sam_eval_metrics_{file_no:06d}.csv',
                    index=False
                )
                print(f"Checkpoint saved at {idx + 1} images, file {file_no}")
                file_no += 1
                results = []

            # Clear GPU cache periodically
            if (idx + 1) % 50 == 0:
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"Error: {json_path.name}: {e}")
            continue

    # Save remaining results
    if results:
        pd.DataFrame(results).to_csv(
            f'/content/drive/MyDrive/sam_dataset/metrics/sam_eval_metrics_{file_no:06d}.csv',
            index=False
        )

    return pd.DataFrame(results)

# Run evaluation
print("Starting SAM evaluation on SA-1B data...")

df = evaluate_sam_optimized(
    extracted_dir=image_dir,
    predictor=predictor
)

Starting SAM evaluation on SA-1B data...


  0%|          | 100/50630 [09:38<61:53:43,  4.41s/it]

Checkpoint saved at 100 images, file 54


  0%|          | 200/50630 [16:28<68:17:01,  4.87s/it]

Checkpoint saved at 200 images, file 55


...

 22%|██▏       | 11000/50630 [12:26:50<41:17:29,  3.75s/it]

Checkpoint saved at 11000 images, file 163


 22%|██▏       | 11100/50630 [12:33:41<66:15:51,  6.03s/it]

Checkpoint saved at 11100 images, file 164


 22%|██▏       | 11165/50630 [12:37:49<40:29:41,  3.69s/it]